In [ ]:
# libraries
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader

# custom
from model import AutoencoderFC
import utilities
from datasets.dataset_IMADS import IMADSDatasetTest

In [ ]:
params = utilities.load_yaml_params()

# Set the seed for MPS torch operations (ones that happen on the MPS Apple GPU)
if params['device'] == 'mps':
    torch.mps.manual_seed(params['seed'])
elif params['device'] == 'cuda':
    torch.cuda.manual_seed(params['seed'])
elif params['device'] == 'cpu':
    torch.manual_seed(params['seed'])
else:
    raise ValueError(f"Wrong device value: {params['device']}")


## Test on pretrained model

In [ ]:
test_dataset = IMADSDatasetTest(machine=params['machine'], window_size_ms=params['window_size_ms'], params=params)
test_data_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)

# Extract the number of channels and window lengths for each sensor\n",
num_channels = [x.shape[1] for x in test_dataset.X]
window_lengths = [x.shape[2] for x in test_dataset.X]
sensors = test_dataset.sensor_dict

model = AutoencoderFC(window_lengths, num_channels, sensors)

# Load checkpoint
checkpoint = torch.load(params['checkpoint_filepath'])
model.load_state_dict(checkpoint['model_state_dict'])


In [ ]:
sensors

In [ ]:
# Test the model sample by sample
anomaly_scores_df, flattened_inputs, predictions, embeddings = model.get_anomaly_scores(test_data_loader, params['criterion'])

# Embeddings analysis

In [ ]:
from pandas.api.types import CategoricalDtype

df  =  pd.DataFrame(embeddings, columns=[f'emb_{k}' for k in range(len(embeddings.T))])
df['segment_id']= anomaly_scores_df['segment_id']
df['anomaly_label'] = anomaly_scores_df['anomaly_label']

Y_test_grouped = utilities.group_by_segment_id(
        df, df.columns[:-2], 'median')


In [ ]:
# Plotting all embeddings against each other
num_embeddings = embeddings.shape[1]
fig, axes = plt.subplots(num_embeddings, num_embeddings, figsize=(15, 15))
fig.suptitle('Scatter Plot of All Embeddings Against Each Other', fontsize=16)
c = Y_test_grouped.anomaly_label.apply(lambda x: 0 if x=='normal' else 1)

for i in range(num_embeddings):
    for j in range(num_embeddings):
        if i != j:
            axes[i, j].scatter(Y_test_grouped[f'emb_{i}'], Y_test_grouped[f'emb_{j}'], c=c, cmap='viridis', s=0.25)
        else:
            axes[i, j].text(0.5, 0.5, f'emb_{i}', ha='center', va='center', fontsize=12)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        axes[i, j].set_xticklabels([])
        axes[i, j].set_yticklabels([])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
plt.figure(figsize=[11, 11])
c = Y_test_grouped.anomaly_label.apply(lambda x: 0 if x=='normal' else 1)

df = Y_test_grouped.drop(['segment_id', 'anomaly_label'], axis=1)
means = df.values.mean(axis=1)
stds = df.values.std(axis=1)

plt.scatter(means, stds, c=c, s=3)

# Losses distributions analysis

In [ ]:
import importlib
importlib.reload(utilities)
from pandas.api.types import CategoricalDtype
import seaborn as sns


def group_and_reorder(df):
       anomaly_scores_col = ['f_imp23absu_mic', 'f_ism330dhcx_acc', 'f_ism330dhcx_gyro',
              's_imp23absu_mic', 's_ism330dhcx_acc', 's_ism330dhcx_gyro',
              'total_loss']

       Y_test_grouped = utilities.group_by_segment_id(
              df, anomaly_scores_col, 'median')
       
       new_order = ['f_imp23absu_mic', 'f_ism330dhcx_acc',
       'f_ism330dhcx_gyro', 's_imp23absu_mic', 's_ism330dhcx_acc',
       's_ism330dhcx_gyro', 'total_loss', 'split_label', 'anomaly_label',
       'domain_shift_op', 'domain_shift_env', 'combined_label','segment_id']

       df = Y_test_grouped[new_order]
       split_labels_order = CategoricalDtype(categories=['Normal_Source_Test', 'Normal_Target_Test', 'Anomaly_Source_Test', 'Anomaly_Target_Test'], ordered=True)
       df['split_label'] = df['split_label'].astype(split_labels_order)

       df = df.sort_values(by='split_label').reset_index(drop=True)
       return df

def reorder(df):

       split_labels_order = CategoricalDtype(categories=['Normal_Source_Test', 'Normal_Target_Test', 'Anomaly_Source_Test', 'Anomaly_Target_Test'], ordered=True)
       df['split_label'] = df['split_label'].astype(split_labels_order)

       df = df.sort_values(by='split_label').reset_index(drop=True)
       return df

def get_violin(df):
       # Generate subplots
       num_features = 7  # Exclude the 'Category' column
       num_columns = 3
       num_rows = (num_features + num_columns - 1) // num_columns  # Calculate the number of rows needed

       fig, axes = plt.subplots(num_rows, num_columns, figsize=(15, 5 * num_rows))

       for i, column in enumerate(df.columns[:-1]):  # Exclude the 'Category' column
              row = i // num_columns
              col = i % num_columns
              if row*num_rows + col == num_features:
                     break

              ax = axes[row, col]
              sns.violinplot(data=df, y=column, hue='split_label', palette='viridis', ax=ax)
              ax.set_title(f'Plot of {column}')
              ax.set_xlabel('sample index')
              ax.set_ylabel(f"Loss - {params['criterion'].__name__}")
              ax.set_xticks(ax.get_xticks())
              ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

       # Remove empty subplots
       for j in range(i + 1, num_rows * num_columns):
              fig.delaxes(axes.flatten()[j])

       plt.tight_layout()
       plt.show()


# Visualize anomaly scores  distribution

In [ ]:
anomaly_scores_df_reordered = reorder(anomaly_scores_df)
get_violin(anomaly_scores_df_reordered)


In [ ]:
anomaly_scores_df_grouped_reordered = group_and_reorder(anomaly_scores_df)
get_violin(anomaly_scores_df_grouped_reordered)

# Thresholding Mechanism for real time inference

## Plot ROC 

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

df = anomaly_scores_df_grouped_reordered

X = df[['total_loss']]
y = df['anomaly_label'].apply(lambda x: 0 if x=='normal' else 1)

fpr, tpr, thresholds = roc_curve(y, X)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

## Plot metrics based on the threshold

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

DECISION = 0.85

# Initialize variables to store the best threshold and its corresponding recall
recalls = []
precisions = []
f1_scores = []
cms = []

# Iterate through thresholds to find the one with the highest recall
for threshold in thresholds:
    y_pred = (X > threshold).astype(int)
    recalls.append(recall_score(y, y_pred))
    precisions.append(precision_score(y, y_pred))
    f1_scores.append(f1_score(y, y_pred))
    cms.append(confusion_matrix(y,y_pred))

decision = np.ones(len(thresholds))*DECISION
scores_df = pd.DataFrame(data = {
    'thresholds': thresholds,
    'recall': recalls,
    'precision': precisions,
    'f1_score': f1_scores
})

scores_df.plot()
plt.plot(decision, 'r-')
plt.legend()

### Prioritize Recall over precision, while keeping both over 85

In [ ]:
# Initialize variables to store the best threshold and its corresponding recall
best_threshold = thresholds[0]
best_recall = 0

# Iterate through thresholds to find the one with the highest recall
for threshold in thresholds:
    y_pred = (X > threshold).astype(int)
    recall = recall_score(y, y_pred)
    precision = precision_score(y, y_pred)
    if recall > best_recall and precision> DECISION:
        best_recall = recall
        best_threshold = threshold

# Display the optimal threshold
print(f"Optimal Threshold: {best_threshold}")

# Calculate the confusion matrix, precision, and recall using the best threshold
y_pred = (X > best_threshold).astype(int)

conf_matrix = confusion_matrix(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1_score_ = f1_score(y, y_pred)

# Display the results
print("Confusion Matrix:")
print(conf_matrix)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1_score_:.2f}")

In [ ]:
import json 
# Save to CSV
results_path = os.path.join(params['results_folder'], params['machine'])
os.makedirs(results_path, exist_ok=True)

df = pd.DataFrame(data = conf_matrix)
df.to_csv(results_path + os.sep + 'conf_matrix.csv')

df = pd.DataFrame([precision,recall,f1_score_], columns = ['value'], index= ['precision', 'recall', 'f1_score'])
df.to_csv(results_path + os.sep + 'metrics.csv')

with open(os.path.join(results_path, 'best_threshold.txt'), 'w') as f:
    f.write(f'best_threshold: {best_threshold}\n')
    params_txt = params.copy()
    params_txt['criterion'] = params_txt['criterion'].__name__
    for k,v in params_txt.items():
        f.write(f'{k}: {v}\n')

